## 1. ติดตั้ง Dependencies

In [ ]:
# !pip install -q --upgrade git+https://github.com/huggingface/transformers.git
# !pip install -q accelerate bitsandbytes pillow pandas tqdm huggingface_hub
!pip install -q ultralytics pandas tqdm pillow scikit-learn
!pip install -q ultralytics pandas tqdm pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00


## 2. Login HuggingFace (ต้องมี token ที่มีสิทธิ์ access Gemma)

In [ ]:
# from huggingface_hub import login

# # ใส่ HuggingFace Token ของคุณที่นี่
# # สร้างได้ที่ https://huggingface.co/settings/tokens
# # ต้องกด 'Agree' terms บน https://huggingface.co/google/gemma-4-e4b-it ก่อน
# HF_TOKEN = ""  # <-- ใส่ token ของคุณ

# login(token=HF_TOKEN)

## 3. Mount Google Drive & ตั้งค่า Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ===== ปรับ PATH ให้ตรงกับโครงสร้างไฟล์ใน Drive ของคุณ =====
BASE_DIR = "/content/drive/MyDrive/super-ai-engineer-season-6-house-recognition-2569"  # โฟลเดอร์หลัก

TRAIN_DIR = os.path.join(BASE_DIR, "train/train")       # โฟลเดอร์รูปภาพ train
TEST_DIR  = os.path.join(BASE_DIR, "test/test")         # โฟลเดอร์รูปภาพ test
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")         # CSV labels ของ train (ถ้ามี)
SAMPLE_CSV= os.path.join(BASE_DIR, "sample_submission.csv")
OUTPUT_CSV= os.path.join(BASE_DIR, "submission.csv")



print(f"Train dir exists : {os.path.exists(TRAIN_DIR)}")
print(f"Test  dir exists : {os.path.exists(TEST_DIR)}")
print(f"Sample CSV exists: {os.path.exists(SAMPLE_CSV)}")

Train dir exists : True
Test  dir exists : True
Sample CSV exists: True


## 4. ตรวจสอบข้อมูล

In [ ]:
import pandas as pd

# อ่าน sample submission
sample_df = pd.read_csv(SAMPLE_CSV)
print(f"Sample submission shape: {sample_df.shape}")
print(sample_df.head(10))
print(f"\nTotal test images expected: {len(sample_df)}")

Sample submission shape: (1550, 2)
         id  answer
0  e4b420b0     0.0
1  23efa479     0.0
2  1f0f2402     0.0
3  8a60480c     NaN
4  11f20127     NaN
5  16173ced     NaN
6  9c05fea1     NaN
7  11f04229     NaN
8  c055fbb7     NaN
9  ec045077     NaN

Total test images expected: 1550


In [ ]:
# ตรวจสอบจำนวนไฟล์ใน test directory
import glob

test_images = sorted(glob.glob(os.path.join(TEST_DIR, "*")))
valid_exts  = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
test_images = [f for f in test_images if os.path.splitext(f)[1] in valid_exts]

print(f"Test images found: {len(test_images)}")
print("ตัวอย่าง:", [os.path.basename(f) for f in test_images[:3]])

Test images found: 1550
ตัวอย่าง: ['00162f19.jpg', '004c4789.jpg', '0059b42f.jpg']


## 5. โหลด Gemma 4 E4B (Multimodal)

In [ ]:
# import torch
# from transformers import AutoProcessor, AutoModelForImageTextToText

# MODEL_ID = "google/gemma-4-e4b-it"

# print(f"Loading processor from {MODEL_ID}...")
# processor = AutoProcessor.from_pretrained(
#     MODEL_ID,
#     token=HF_TOKEN
# )

# print(f"Loading model from {MODEL_ID}...")
# model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID,
#     token=HF_TOKEN,
#     torch_dtype=torch.bfloat16,   # ใช้ bf16 เพื่อประหยัด VRAM
#     device_map="auto",            # auto map ไปยัง GPU
# )
# model.eval()

# print("✅ Model loaded successfully!")
# print(f"Device: {next(model.parameters()).device}")

In [ ]:
# import shutil
# from sklearn.model_selection import train_test_split
# from ultralytics import YOLO

# YOLO_DATASET = TRAIN_DIR

# # อ่าน train labels
# train_df = pd.read_csv(TRAIN_CSV)

# # แบ่ง train/val 90/10
# train_split, val_split = train_test_split(
#     train_df, test_size=0.1, random_state=42, stratify=train_df["answer"]
# )

# # สร้าง folder structure
# for split in ["train", "val"]:
#     for label in ["0", "1"]:
#         os.makedirs(os.path.join(YOLO_DATASET, split, label), exist_ok=True)

# # Copy รูปเข้า folder
# def copy_images(df, split_name):
#     for _, row in df.iterrows():
#         src = find_image_path(str(row["id"]), TRAIN_DIR)
#         if src:
#             dst = os.path.join(YOLO_DATASET, split_name, str(int(row["answer"])))
#             shutil.copy2(src, dst)

# copy_images(train_split, "train")
# copy_images(val_split, "val")
# print("✅ Dataset ready!")

# # Train YOLO
# model = YOLO("yolov8m-cls.pt")
# model.train(
#     data=YOLO_DATASET,
#     epochs=20,
#     imgsz=224,
#     batch=32,
#     device=0,
#     patience=5,
#     project="/content/runs",
#     name="house_cls",
#     exist_ok=True,
# )

# # โหลด best model
# model = YOLO("/content/runs/house_cls/weights/best.pt")
# print(f"✅ Best model loaded | Classes: {model.names}")

In [ ]:
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from tqdm.auto import tqdm
import pandas as pd
import os


YOLO_DATASET = "/content/yolo_dataset"

# อ่าน train labels
train_df = pd.read_csv(TRAIN_CSV)

# แบ่ง train/val 90/10
train_split, val_split = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df["class"]
)

# สร้าง folder structure
for split in ["train", "val"]:
    for label in ["0", "1"]:
        os.makedirs(os.path.join(YOLO_DATASET, split, label), exist_ok=True)

# Copy รูปเข้า folder
def copy_images(df, split_name):
    found, missing = 0, 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {split_name}"):
        src = os.path.join(TRAIN_DIR, row["image_name"])
        if os.path.exists(src):
            dst = os.path.join(YOLO_DATASET, split_name, str(int(row["class"])))
            shutil.copy2(src, dst)
            found += 1
        else:
            missing += 1
    print(f"  ✅ {split_name}: copied {found} | missing {missing}")

copy_images(train_split, "train")
copy_images(val_split, "val")
print("✅ Dataset ready!")

# Train YOLO - ปรับเป็นรุ่น Nano และใช้ CPU
model = YOLO("yolov8m-cls.pt")
model.train(
    data=YOLO_DATASET,
    epochs=25,         # ลดจำนวน Epoch ลงเพื่อให้เสร็จเร็วขึ้นบน CPU
    imgsz=128,        # ลดขนาดรูปภาพลงเล็กน้อยเพื่อประหยัดแรง CPU
    batch=16,
    device="cpu",     # กำหนดเป็น cpu
    project="/content/runs",
    name="house_cls",
    exist_ok=True,
)

# โหลด best model
model = YOLO("/content/runs/house_cls/weights/best.pt")
print(f"✅ Best model loaded | Classes: {model.names}")

## 6. ทดสอบ Inference ด้วยรูปตัวอย่าง

In [ ]:
# from PIL import Image
# import re

# # ===== Prompt สำหรับ Classification =====
# SYSTEM_PROMPT = """You are an expert at analyzing street-view images of residential properties in Thailand.
# Your task is to determine whether the TARGET house/building is clearly visible and is the main subject of the image.

# Answer ONLY with 'YES' or 'NO'.
# - YES: The target residential building is clearly visible and is the main/primary subject.
# - NO: The image shows only neighboring buildings, empty land, trees, roads, or the target building is completely obstructed."""

# USER_PROMPT = """Look at this street-view image. Is there a clear residential house or building that is the main subject of this photo?

# Important rules:
# 1. If the image shows a house/building clearly as the main subject → YES
# 2. If the image shows only neighboring buildings with no clear primary house → NO
# 3. If the image shows only trees, empty land, fence without visible house → NO
# 4. Commercial buildings, shops, apartments count as YES if they are the clear primary subject

# Answer with only YES or NO:"""


# def classify_image(image_path: str) -> int:
#     """
#     ส่งรูปเข้า Gemma 4 แล้วให้ตอบ YES/NO
#     คืนค่า 1 (YES) หรือ 0 (NO)
#     """
#     try:
#         image = Image.open(image_path).convert("RGB")
#     except Exception as e:
#         print(f"Error loading {image_path}: {e}")
#         return 0

#     # สร้าง conversation format ของ Gemma 4
#     messages = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "image", "image": image},
#                 {"type": "text",  "text": USER_PROMPT},
#             ],
#         }
#     ]

#     # ใช้ apply_chat_template ของ Gemma 4
#     text = processor.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )

#     inputs = processor(
#         text=[text],
#         images=[image],
#         return_tensors="pt",
#         padding=True,
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=10,
#             do_sample=False,
#             temperature=None,
#             top_p=None,
#         )

#     # Decode เฉพาะ token ที่ generate ใหม่
#     input_len = inputs["input_ids"].shape[1]
#     generated = outputs[0][input_len:]
#     response  = processor.decode(generated, skip_special_tokens=True).strip().upper()

#     # Parse คำตอบ
#     if "YES" in response:
#         return 1
#     elif "NO" in response:
#         return 0
#     else:
#         # fallback: ถ้าตอบไม่ชัดเจน ให้ดูตัวอักษรแรก
#         print(f"  ⚠️ Ambiguous response: '{response}' → defaulting to 0")
#         return 0


# # ทดสอบกับรูปตัวอย่าง (ถ้ามี)
# if test_images:
#     sample_path = test_images[0]
#     result = classify_image(sample_path)
#     print(f"Test image: {os.path.basename(sample_path)}")
#     print(f"Prediction: {result} ({'YES (บ้าน)' if result == 1 else 'NO (ไม่ใช่บ้าน)'})")

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test image: 00162f19.jpg
Prediction: 0 (NO (ไม่ใช่บ้าน))


In [ ]:
def classify_image(image_path: str) -> int:
    result = model.predict(image_path, verbose=False)[0]
    pred_class_name = model.names[result.probs.top1]  # '0' หรือ '1'
    return int(pred_class_name)

## 7. Run Inference บน Test Set ทั้งหมด (1550 รูป)

In [ ]:
from tqdm.auto import tqdm
import time

# อ่าน sample submission เพื่อให้แน่ใจว่า ID ตรงกัน
sample_df = pd.read_csv(SAMPLE_CSV, dtype=str)
test_ids  = sample_df["id"].astype(str).tolist()

print(f"Total IDs to classify: {len(test_ids)}")

# # Map ID → file path
# # รูปภาพชื่อไฟล์อาจมีหลาย extension ลอง .jpg, .jpeg, .png
# def find_image_path(img_id: str, test_dir: str) -> str | None:
#     for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
#         p = os.path.join(test_dir, img_id + ext)
#         if os.path.exists(p):
#             return p
#     # บางไฟล์อาจมีชื่อยาวกว่า (เช่น ChokChai4_img_...)
#     # ลอง glob หา prefix match
#     matches = glob.glob(os.path.join(test_dir, img_id + "*"))
#     if matches:
#         return matches[0]
#     return None

def find_image_path(img_id: str, test_dir: str):
    # ลองหาชื่อตรงก่อน
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = os.path.join(test_dir, img_id + ext)
        if os.path.exists(p):
            return p
    # ถ้าไม่เจอ → ค้นหา ID ที่อยู่กลางชื่อไฟล์ด้วย **wildcard ทั้งสองด้าน**
    matches = glob.glob(os.path.join(test_dir, f"*{img_id}*"))
    return matches[0] if matches else None

# ทดสอบ lookup
sample_path = find_image_path(test_ids[0], TEST_DIR)
print(f"Lookup test → ID: {test_ids[0]}, Path: {sample_path}")


Total IDs to classify: 1550
Lookup test → ID: e4b420b0, Path: /content/drive/MyDrive/super-ai-engineer-season-6-house-recognition/test/test/e4b420b0.jpg


In [ ]:
# ============================================================
# Main Inference Loop
# ============================================================
predictions = []
missing     = []

start_time = time.time()

for img_id in tqdm(test_ids, desc="Classifying"):
    img_path = find_image_path(str(img_id), TEST_DIR)

    if img_path is None:
        print(f"⚠️  Image not found for ID: {img_id} → defaulting to 0")
        predictions.append(0)
        missing.append(img_id)
    else:
        pred = classify_image(img_path)
        predictions.append(pred)

elapsed = time.time() - start_time
print(f"\n✅ Done! {len(predictions)} predictions in {elapsed/60:.1f} min")
print(f"Missing images : {len(missing)}")
print(f"YES (1): {sum(predictions)} | NO (0): {predictions.count(0)}")

Classifying:   0%|          | 0/1550 [00:00<?, ?it/s]


✅ Done! 1550 predictions in 0.7 min
Missing images : 0
YES (1): 755 | NO (0): 795


## 8. บันทึก Submission CSV

In [ ]:
submission_df = pd.DataFrame({
    "id":     test_ids,
    "answer": predictions,
})

submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Submission saved to: {OUTPUT_CSV}")
print(f"Shape: {submission_df.shape}")
print(submission_df.head(10))
print("\nValue counts:")
print(submission_df["answer"].value_counts())

✅ Submission saved to: /content/drive/MyDrive/super-ai-engineer-season-6-house-recognition/submission.csv
Shape: (1550, 2)
         id  answer
0  e4b420b0       0
1  23efa479       0
2  1f0f2402       0
3  8a60480c       0
4  11f20127       0
5  16173ced       0
6  9c05fea1       0
7  11f04229       0
8  c055fbb7       0
9  ec045077       0

Value counts:
answer
0    795
1    755
Name: count, dtype: int64


In [ ]:
# เพิ่ม dtype=str เพื่อป้องกัน pandas แปลงเป็น scientific notation
sample_df = pd.read_csv(SAMPLE_CSV, dtype=str)
test_ids  = sample_df["id"].tolist()

test_ids

['e4b420b0',
 '23efa479',
 '1f0f2402',
 '8a60480c',
 '11f20127',
 '16173ced',
 '9c05fea1',
 '11f04229',
 'c055fbb7',
 'ec045077',
 '1310e8df',
 'baf16c49',
 '54d1ba78',
 '5e8aea48',
 '35d7632c',
 'd706d008',
 '3e5a0ca6',
 'b69f38f2',
 '582c787c',
 'e628fac1',
 '11d6e00c',
 '4d1b8a1e',
 'ce27fd73',
 '635b361b',
 '504c8cfe',
 '583e01c9',
 '9e4334b4',
 '7ca0a84e',
 'e0d17db9',
 'b233ede2',
 '004c4789',
 '91ea5755',
 '3f07beda',
 '63d13ecb',
 'fb6fe557',
 '45d1d0da',
 'edf66b63',
 '7ce8fc51',
 '6d626e6c',
 '5af5198a',
 '7ba6d581',
 '34064367',
 '2fe4a495',
 '73729abb',
 'e69c40e0',
 'e0eb1696',
 '16226aef',
 '1e98c362',
 '52bfa100',
 '7e8862e8',
 '4cd45008',
 'c9d1a911',
 'ce937c2f',
 '82402242',
 '0e250170',
 'b1c66306',
 '2b3d31e5',
 '03df66ca',
 '728caeea',
 'a4418dbf',
 '99fab84f',
 '14797aa9',
 '32d34d03',
 '438de536',
 '5c59c118',
 'a69b32bc',
 '83235a6b',
 'd54e3433',
 'a8261e6a',
 '25b85661',
 'c87aa242',
 '7d7a3d8c',
 '0188f57d',
 '04dd10ad',
 '449b791c',
 'e9d3a97d',
 'd7678327',

## 9. (Optional) ตรวจสอบ Format ให้ตรงกับ Sample

In [ ]:
# ตรวจสอบว่า submission มีครบทุก ID และ format ถูกต้อง
sample_ids = set(sample_df["id"].astype(str).tolist())
submit_ids = set(submission_df["id"].astype(str).tolist())

missing_in_submit = sample_ids - submit_ids
extra_in_submit   = submit_ids - sample_ids

print(f"IDs in sample  : {len(sample_ids)}")
print(f"IDs in submit  : {len(submit_ids)}")
print(f"Missing IDs    : {len(missing_in_submit)}")
print(f"Extra IDs      : {len(extra_in_submit)}")

assert submission_df["answer"].isin([0, 1]).all(), "⚠️ answer ต้องเป็น 0 หรือ 1 เท่านั้น!"
print("\n✅ Format check passed! พร้อม submit")

IDs in sample  : 1550
IDs in submit  : 1550
Missing IDs    : 0
Extra IDs      : 0

✅ Format check passed! พร้อม submit


## 10. (Bonus) Few-Shot Prompting เพื่อเพิ่ม Accuracy

ถ้า accuracy ยังต่ำ ลอง Few-Shot โดยส่งรูปตัวอย่างเข้าไปด้วย

In [ ]:
# ===== Few-Shot Version =====
# โหลดรูป positive/negative ตัวอย่างจาก training set

def classify_image_fewshot(
    image_path: str,
    pos_example_path: str,  # ตัวอย่าง YES (บ้าน)
    neg_example_path: str,  # ตัวอย่าง NO (ไม่ใช่บ้าน)
) -> int:
    """
    Few-shot version: ส่ง 2 ตัวอย่างก่อนแล้วค่อยถาม
    """
    try:
        image     = Image.open(image_path).convert("RGB")
        pos_image = Image.open(pos_example_path).convert("RGB")
        neg_image = Image.open(neg_example_path).convert("RGB")
    except Exception as e:
        print(f"Error: {e}")
        return 0

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image",  "image": neg_image},
                {"type": "text",   "text": "Is there a house clearly visible as the main subject?"},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": "NO"}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image",  "image": pos_image},
                {"type": "text",   "text": "Is there a house clearly visible as the main subject?"},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": "YES"}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image",  "image": image},
                {"type": "text",   "text": "Is there a house clearly visible as the main subject? Answer only YES or NO:"},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    all_images = [neg_image, pos_image, image]

    inputs = processor(
        text=[text],
        images=all_images,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    input_len = inputs["input_ids"].shape[1]
    generated = outputs[0][input_len:]
    response  = processor.decode(generated, skip_special_tokens=True).strip().upper()

    return 1 if "YES" in response else 0


print("Few-shot function defined ✅")
print("วิธีใช้: ตั้งค่า pos_example_path / neg_example_path จาก training set แล้วแทน classify_image ด้วย classify_image_fewshot")

Few-shot function defined ✅
วิธีใช้: ตั้งค่า pos_example_path / neg_example_path จาก training set แล้วแทน classify_image ด้วย classify_image_fewshot


## 11. (Optional) Validate บน Training Set เพื่อดู Accuracy

In [ ]:
# ถ้ามี train_labels.csv ลองวัด accuracy บน subset ก่อน submit

VALIDATE = False  # ตั้งเป็น True ถ้าต้องการ validate
VALIDATE_N = 100  # จำนวนตัวอย่างที่จะใช้ validate

if VALIDATE and os.path.exists(TRAIN_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    print(f"Train data shape: {train_df.shape}")
    print(train_df.head())

    # สุ่ม N ตัวอย่าง
    val_sample = train_df.sample(n=VALIDATE_N, random_state=42)

    val_preds = []
    val_labels = []

    for _, row in tqdm(val_sample.iterrows(), total=len(val_sample), desc="Validating"):
        img_id    = str(row["id"])
        true_label = int(row["answer"])
        img_path   = find_image_path(img_id, TRAIN_DIR)

        if img_path:
            pred = classify_image(img_path)
            val_preds.append(pred)
            val_labels.append(true_label)

    # คำนวณ Accuracy
    accuracy = sum(p == l for p, l in zip(val_preds, val_labels)) / len(val_labels)
    print(f"\n✅ Validation Accuracy ({len(val_labels)} samples): {accuracy:.4f} ({accuracy*100:.2f}%)")
else:
    print("Skipping validation (VALIDATE=False หรือไม่มี train_labels.csv)")

Skipping validation (VALIDATE=False หรือไม่มี train_labels.csv)
